In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
# CEP é lido como string: como inteiro os zeros à esquerda se perdem (ex.: 01046 -> 1046)
zip_dtypes = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}

datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file, dtype=zip_dtypes)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframes
translation_df = pd.DataFrame(datasets["product_category_name_translation"])
products_clean = pd.read_csv("../data/processed/olist_products_dataset_clean.csv")
translation_df.shape

(71, 2)

In [6]:
translation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [7]:
#Avaliando se existe valores nulos e duplicados
print(f"Valores nulos: {translation_df.isna().sum().sum()}")
print(f"Linhas duplicadas: {translation_df.duplicated().sum()}")
print(f"product_category_name duplicados: {translation_df['product_category_name'].duplicated().sum()}")
print(f"Traduções em inglês duplicadas: {translation_df['product_category_name_english'].duplicated().sum()}")

Valores nulos: 0
Linhas duplicadas: 0
product_category_name duplicados: 0
Traduções em inglês duplicadas: 0


In [8]:
# Padrão dos nomes: minúsculo, sem espaços (snake_case)
print(f"Nomes fora do padrão (português): {translation_df['product_category_name'].str.contains(r'[A-Z\s]').sum()}")
print(f"Nomes fora do padrão (inglês): {translation_df['product_category_name_english'].str.contains(r'[A-Z\s]').sum()}")

Nomes fora do padrão (português): 0
Nomes fora do padrão (inglês): 0


In [9]:
# Categorias de products (já tratado) que não existem na tabela de tradução
missing_categories = sorted(set(products_clean['product_category_name']) - set(translation_df['product_category_name']))
print(missing_categories)

products_clean['product_category_name'].isin(missing_categories).sum()

['Unknown', 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


np.int64(623)

In [10]:
# Quantidade de produtos afetados por categoria sem tradução
products_clean.loc[products_clean['product_category_name'].isin(missing_categories), 'product_category_name'].value_counts()

product_category_name
Unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64

In [11]:
#Fazendo copia do dataframe para não alterar o original
translation_clean = translation_df.copy()

In [12]:
# Adicionando as categorias sem tradução
# 'Unknown' é o valor usado em products para produtos sem categoria
new_categories = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "Unknown"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_preparers", "unknown"],
})

translation_clean = pd.concat([translation_clean, new_categories], ignore_index=True)
translation_clean.tail()

,product_category_name,product_category_name_english
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services
71,pc_gamer,pc_gamer
72,portateis_cozinha_e_preparadores_de_alimentos,portable_kitchen_food_preparers
73,Unknown,unknown


In [13]:
# Validações pós-tratamento
print(f"Linhas: {translation_clean.shape[0]} (original: {translation_df.shape[0]})")
print(f"Valores nulos: {translation_clean.isna().sum().sum()}")
print(f"product_category_name duplicados: {translation_clean['product_category_name'].duplicated().sum()}")
print(f"Categorias de products sem tradução: {(~products_clean['product_category_name'].isin(translation_clean['product_category_name'])).sum()}")

# Valores originais permanecem iguais
print(f"Linhas originais preservadas: {translation_clean.iloc[:len(translation_df)].equals(translation_df)}")

assert translation_clean['product_category_name'].is_unique
assert products_clean['product_category_name'].isin(translation_clean['product_category_name']).all()

Linhas: 74 (original: 71)
Valores nulos: 0
product_category_name duplicados: 0
Categorias de products sem tradução: 0
Linhas originais preservadas: True


In [14]:
# Salvando dataframe tratado em pasta processed
translation_clean.to_csv('../data/processed/product_category_name_translation_clean.csv', index=False)

## 📌 Conclusão da etapa — Category Translation

### 🔎 O que foi encontrado?

A tabela `product_category_name_translation` possui **71 registros e 2 colunas**, sem nulos, sem duplicidade e com nomes no padrão `snake_case`.

Ao cruzar com `products` (já tratado), **623 produtos** têm categoria sem tradução:

- `Unknown` (**610** produtos, valor usado na limpeza de `products` para categoria ausente);
- `portateis_cozinha_e_preparadores_de_alimentos` (**10** produtos);
- `pc_gamer` (**3** produtos).

### 🛠️ O que foi feito?

Foram adicionadas **3 linhas** de tradução:

| product_category_name | product_category_name_english |
|---|---|
| `pc_gamer` | `pc_gamer` |
| `portateis_cozinha_e_preparadores_de_alimentos` | `portable_kitchen_food_preparers` |
| `Unknown` | `unknown` |

As 71 linhas originais não foram alteradas.

### ✅ Como foi validado?

- A tabela passou de 71 para **74 registros**, sem nulos e sem `product_category_name` duplicado;
- Linhas originais idênticas às da tabela bruta;
- **Todas as categorias de `products` possuem tradução** (0 sem correspondência).

### 📁 Resultado

O dataset tratado foi salvo em:

`data/processed/product_category_name_translation_clean.csv`